<a href="https://colab.research.google.com/github/niya-c-anto/ASSIGNMENTS/blob/main/Resnet18_and_mobilenet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# --------------------------
# 1. Device
# --------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------
# 2. Data
# --------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # ResNet expects 224x224
    transforms.Grayscale(num_output_channels=3), # convert 1->3 channels
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# --------------------------
# 3. Model
# --------------------------
resnet = models.resnet18(pretrained=False)

# Change output layer to 10 classes (MNIST)
resnet.fc = nn.Linear(resnet.fc.in_features, 10)
resnet = resnet.to(device)

# --------------------------
# 4. Loss & Optimizer
# --------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet.parameters(), lr=0.001)

# --------------------------
# 5. Training Loop
# --------------------------
epochs = 5

for epoch in range(epochs):
    resnet.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = resnet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

# --------------------------
# 6. Evaluation
# --------------------------
resnet.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = resnet(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")


100%|██████████| 9.91M/9.91M [00:00<00:00, 18.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 507kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.50MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.6MB/s]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch [1/5], Loss: 0.1030
Epoch [2/5], Loss: 0.0412
Epoch [3/5], Loss: 0.0323
Epoch [4/5], Loss: 0.0265
Epoch [5/5], Loss: 0.0241
Test Accuracy: 99.15%


In [2]:
# --------------------------
# 1. Model
# --------------------------
mobilenet = models.mobilenet_v2(pretrained=False)

# Modify first layer for grayscale (1 channel)
mobilenet.features[0][0] = nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False)  # MNIST images converted to 3 channels

# Change classifier to 10 classes
mobilenet.classifier[1] = nn.Linear(mobilenet.classifier[1].in_features, 10)
mobilenet = mobilenet.to(device)

# --------------------------
# 2. Loss & Optimizer
# --------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mobilenet.parameters(), lr=0.001)

# --------------------------
# 3. Training Loop (same as ResNet)
# --------------------------
for epoch in range(epochs):
    mobilenet.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = mobilenet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

# --------------------------
# 4. Evaluation
# --------------------------
mobilenet.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = mobilenet(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")


Epoch [1/5], Loss: 0.1575
Epoch [2/5], Loss: 0.0490
Epoch [3/5], Loss: 0.0397
Epoch [4/5], Loss: 0.0346
Epoch [5/5], Loss: 0.0321
Test Accuracy: 98.92%
